In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm
from scipy.stats import spearmanr
import functools, operator
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import (
    load_config, load_variant_class, scan_variants, appv_of, filter_covered,
    pick_annos, gene_trait_tool_correlations, env_override, fetch_hf_data,
)


In [ ]:
# ===================== Parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override) =====================
variant_class       = env_override('VARIANT_CLASS', 'missense')    # defines variant filters + tool set
# 'gnomad' excluded by default: PROTEINGYM_FILE has no gnomAD AF columns (lab-generated DMS variants, not
# variants observed in real genomes). Within 'missense', polyphen/sift are also absent from that
# file (never joined) and get silently dropped by the existing_annos schema-intersection below --
# so the common tool set here is 12, not the full 16 used by correlations.ipynb /
# clinvar_spearman_scatterplot.ipynb. Real data-availability limit, not filtering drift.
selected_categories = env_override('SELECTED_CATEGORIES',
                                    ['missense', 'conservation', 'genetic_diversity'], 'list')

mac                 = env_override('MAC', 20, int)                  # rare-variant cap (phenotype correlations)
only_snps           = env_override('ONLY_SNPS', True, bool)
exclude_clinvar     = env_override('EXCLUDE_CLINVAR', False, bool)
only_clinvar        = env_override('ONLY_CLINVAR', False, bool)

# ===================== Configs =====================
CFG = str(REPO_ROOT / 'configs')
anno_config_df, all_annotation_list = load_config(CFG, 'config_correlations.yaml')
vc = load_variant_class(CFG, variant_class)

# Tools (annotations) evaluated in BOTH analyses
selected_annos = anno_config_df.filter(pl.col('category').is_in(selected_categories))['annotation'].to_list()


In [ ]:
# ===================== Local data paths =====================
# Association/correlation files are not read here: the master table already carries
# phenotype + loftee_corr_dir per gene (see genebass/README.md, master-table build step 1),
# so gene_trait_tool_correlations() below reads them straight off MASTER_PATH.
MASTER_PATH     = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
PROTEINGYM_FILE = env_override('PROTEINGYM_PATH', fetch_hf_data('other_benchmarks/proteingym_snv_annotated.parquet', REPO_ROOT))
FIG_DIR         = env_override('FIG_DIR', '../../../paper_figures')
MIN_VARIANTS_PER_GENE = env_override('MIN_VARIANTS', 100, int)   # (gene, tool) pairs with fewer scored variants are dropped

print(f"variant_class = {variant_class}")
print(f"categories    = {selected_categories}")
print(f"{len(selected_annos)} tools: {selected_annos}")
anno_config_df.filter(pl.col('annotation').is_in(selected_annos))


# ProteinGym correlations

In [ ]:
pgdf = pl.read_parquet(PROTEINGYM_FILE)

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in pgdf.columns]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
print(selected_annos)

pgdf

In [ ]:
pg_melt = (
    pgdf
    .select(
        set(['file_name', 'exp_readout', 'mutant', 'id', 'region', 'dms_score']).union(set(selected_annos))
    )
    .unpivot(
        index=['file_name', 'exp_readout', 'mutant', 'id', 'region', 'dms_score'],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
)

pg_melt

In [ ]:
pg_corr = (
    pg_melt
    # Spearman: rank with "average" for proper tie handling
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "file_name", "exp_readout", "annotation"])
        .alias(f"{c}_rank")
        for c in ['annotation_score', 'dms_score']
    )

    .group_by(["region", "file_name", "exp_readout", "annotation"])
    .agg(
        n_variants = pl.col("id").count(),
        correlation = pl.when(
            (pl.col("annotation_score_rank").n_unique() > 1) & 
            (pl.col("dms_score_rank").n_unique() > 1)
        )
        .then(
            pl.corr("annotation_score_rank", "dms_score_rank", propagate_nans=True)
        )
        .otherwise(None)
    )

    .select(["region", "file_name", "exp_readout", "annotation", "n_variants", "correlation"])
    
    .drop_nans().drop_nulls()

    .join(
        anno_config_df.filter(pl.col("category").is_in(selected_categories)),
        on='annotation'
    )
    .with_columns(
        corr_dircor = pl.col('correlation')*pl.col('annotation_dir')*-1 # higher the DMS_score value, the higher the fitness of the mutated protein
    )
)

pg_corr

In [ ]:
pg_filt_corr_df = filter_covered(pg_corr.drop_nans(), group_col=["region", "file_name", "exp_readout"],
                                 n_variants_col="n_variants", min_variants=100)
pg_filt_corr_df

In [ ]:
avg_pg_corr_str = (
    pg_filt_corr_df
    .group_by(["annotation", "category", "color", "label", "exp_readout"])
    .agg(
        pg_n_assays = pl.col("file_name").n_unique(),
        pg_n_variants = pl.col("n_variants").sum().round(0).cast(pl.Int32),
        
        pg_mean_corr_dircor = pl.col("corr_dircor").mean(),
        pg_std_corr_dircor = pl.col("corr_dircor").std(),
        pg_sem_corr_dircor = (pl.col("corr_dircor").std() / pl.col("file_name").len().sqrt()),
    )
    .with_columns(
        pg_ci_low = pl.col("pg_mean_corr_dircor") - 1.96 * pl.col("pg_sem_corr_dircor"),
        pg_ci_high = pl.col("pg_mean_corr_dircor") + 1.96 * pl.col("pg_sem_corr_dircor")
    )
)

avg_pg_corr_str

In [ ]:
avg_pg_corr = (
    pg_filt_corr_df
    .group_by(["annotation", "category", "color", "label"])
    .agg(
        pg_n_assays = pl.col("file_name").n_unique(),
        pg_n_variants = pl.col("n_variants").sum().round(0).cast(pl.Int32),

        pg_mean_corr_dircor = pl.col("corr_dircor").mean(),
        pg_std_corr_dircor = pl.col("corr_dircor").std(),
        pg_sem_corr_dircor = (pl.col("corr_dircor").std() / pl.col("file_name").len().sqrt()),
    )
    .with_columns(
        pg_ci_low = pl.col("pg_mean_corr_dircor") - 1.96 * pl.col("pg_sem_corr_dircor"),
        pg_ci_high = pl.col("pg_mean_corr_dircor") + 1.96 * pl.col("pg_sem_corr_dircor")
    )
)

avg_pg_corr

# UKBBGym correlations

In [ ]:
# Same canonical per-(gene, trait, tool) correlation + coverage computation as
# correlations.ipynb / clinvar_spearman_scatterplot.ipynb /
# proteingym_snr.ipynb -- the master table already carries phenotype and
# loftee_corr_dir per gene, so this reads MASTER_PATH directly rather than rebuilding
# gene_trait_df from the separate association/correlation files.
lf = scan_variants(MASTER_PATH, vc, only_snps=only_snps, only_clinvar=only_clinvar,
                    exclude_clinvar=exclude_clinvar)
annos = pick_annos(anno_config_df, all_annotation_list, selected_categories, lf.collect_schema().names())
print(f'{len(annos)} tools: {annos}')

filt_corr_df = gene_trait_tool_correlations(lf, mac, annos, anno_config_df, selected_categories,
                                            MIN_VARIANTS_PER_GENE)
filt_corr_df


In [ ]:
avg_corr = (
    filt_corr_df
    .group_by(["annotation", "category", "color", "label"])
    .agg(
        n_genes = pl.col("region").n_unique(),
        n_variants = pl.col("n_variants").sum().round(0).cast(pl.Int32),

        mean_corr_dircor = pl.col("corr_beta").mean(),
        std_corr_dircor = pl.col("corr_beta").std(),
        sem_corr_dircor = (pl.col("corr_beta").std() / pl.col("region").len().sqrt()),

        # median_corr_dircor = pl.col("corr_beta").median(),
        # quant_low = pl.col("corr_beta").quantile(0.25),
        # quant_high = pl.col("corr_beta").quantile(0.75),
    )
    .with_columns(
        ci_low = pl.col("mean_corr_dircor") - 1.96 * pl.col("sem_corr_dircor"),
        ci_high = pl.col("mean_corr_dircor") + 1.96 * pl.col("sem_corr_dircor")
    )
)

avg_corr

# Merge and make scatter plot

In [ ]:
scatter_df = (
    avg_corr
    .join(
        avg_pg_corr.drop(['category', 'color', 'label']), 
        on='annotation', 
        how='inner'
    )
)

cat_color = dict(
    anno_config_df.filter(pl.col('annotation').is_in(scatter_df['annotation']))
    .group_by('category').agg(pl.col('color').first()).iter_rows()
)

MISSENSE_ANNOS = ["CPT-1", "BayesDel", "REVEL", "ClinPred", "AlphaMissense", "ESM1v", "GPN-MSA", "Vertebrate PhyloP", "CADD Raw"]

LABELED_ANNOS = ['CPT-1', 'GPN-MSA', 'Vertebrate PhyloP']

scatter_df_small = scatter_df.filter(pl.col('label').is_in(MISSENSE_ANNOS))
scatter_df_small = scatter_df_small.with_columns(
    label_display = pl.when(pl.col('label').is_in(LABELED_ANNOS)).then(pl.col('label')).otherwise(pl.lit(''))
)

overall_corr = scatter_df_small.select(
    spearman_r = pl.corr(
        pl.col('pg_mean_corr_dircor').rank('average'),
        pl.col('mean_corr_dircor').rank('average'),
    ),
    x = pl.col('pg_ci_low').min(),
    y = pl.col('ci_high').max(),
).with_columns(
    corr_label = "\u03c1 = " + pl.col('spearman_r').round(2).cast(pl.Utf8)
)

plot = (
    ggplot(scatter_df_small, aes(x='pg_mean_corr_dircor', y='mean_corr_dircor'))
    + geom_errorbar(aes(ymin='ci_low', ymax='ci_high'), width=0.001, color='black')
    + geom_errorbarh(aes(xmin='pg_ci_low', xmax='pg_ci_high'), height=0, color='black')
    + geom_point(aes(fill='category'), color='black', size=4.5, stroke=0.6)
    + geom_text(aes(label='label_display'), color='black', size=11, ha='left', nudge_x=0.008, va='bottom', nudge_y=0.001)
    + geom_text(data=overall_corr, mapping=aes(x='x', y='y', label='corr_label'),
                color='black', size=12, ha='left', va='top', fontweight='bold')
    + scale_fill_manual(values=cat_color, name='Tool category')
    + labs(
        x=f"per-gene ProteinGym correlation\n({scatter_df_small['pg_n_assays'][0]} assays)",
        y=f"per-gene UKBBGym correlation\n({scatter_df_small['n_genes'][0]} genes)",
    )
    + theme_minimal()
    + theme(
        figure_size=(7, 6),
        axis_text=element_text(size=14),
        axis_title=element_text(size=14, lineheight=1.4),
        legend_position=(0.95, 0.05),
        legend_text=element_text(size=14),
        legend_title=element_text(size=14),
        legend_background=element_rect(fill="white", color="white", alpha=1),
        plot_background=element_rect(fill="white", color="white"),
        panel_grid_major=element_line(color="#cccccc", size=0.6),
        panel_grid_minor=element_line(color="#dddddd", size=0.3),
    )
)

plot.save(f"{FIG_DIR}/FS_proteingym_correlation_scatter_overall.svg", dpi=200)

plot

In [ ]:
scatter_df_str = (
    avg_corr
    .join(
        avg_pg_corr_str[['annotation', 'exp_readout', 'pg_n_variants', 'pg_mean_corr_dircor', 'pg_std_corr_dircor', 'pg_sem_corr_dircor', 'pg_n_assays', 'pg_ci_low', 'pg_ci_high']], 
        on='annotation', 
        how='inner'
    )
    .with_columns(
        readout_label = pl.col('exp_readout') + " (" + pl.col('pg_n_assays').cast(pl.Utf8) + " assays" + ")"
    )
)

cat_color = dict(
    anno_config_df.filter(pl.col('annotation').is_in(scatter_df_str['annotation']))
    .group_by('category').agg(pl.col('color').first()).iter_rows()
)

MISSENSE_ANNOS = ["CPT-1", "BayesDel", "REVEL", "ClinPred", "AlphaMissense", "ESM1v", "GPN-MSA", "Vertebrate PhyloP", "CADD Raw"]

LABELED_ANNOS = ['CPT-1', 'GPN-MSA', 'Vertebrate PhyloP']

scatter_df_small = scatter_df_str.filter(pl.col('label').is_in(MISSENSE_ANNOS))
scatter_df_small = scatter_df_small.with_columns(
    label_display = pl.when(pl.col('label').is_in(LABELED_ANNOS)).then(pl.col('label')).otherwise(pl.lit(''))
)

facet_corr = (
    scatter_df_small
    .group_by('readout_label')
    .agg(
        spearman_r = pl.corr(
            pl.col('pg_mean_corr_dircor').rank('average'),
            pl.col('mean_corr_dircor').rank('average'),
        ),
        x = pl.col('pg_ci_low').min(),
        y = pl.col('ci_high').max(),
    )
    .with_columns(
        corr_label = "\u03c1 = " + pl.col('spearman_r').round(2).cast(pl.Utf8)
    )
)

plot = (
    ggplot(scatter_df_small, aes(x='pg_mean_corr_dircor', y='mean_corr_dircor'))
    + geom_errorbar(aes(ymin='ci_low', ymax='ci_high'), width=0, color='black')
    + geom_errorbarh(aes(xmin='pg_ci_low', xmax='pg_ci_high'), height=0, color='black')
    + geom_point(aes(fill='category'), color='black', size=4, stroke=0.6)
    + geom_text(aes(label='label_display'), color='black', size=11, ha='left', nudge_x=0.008, va='bottom', nudge_y=0.001)
    + geom_text(data=facet_corr, mapping=aes(x='x', y='y', label='corr_label'), color='black', size=12, ha='left', va='top', fontweight='bold')
    + scale_fill_manual(values=cat_color, name='Tool category')
    + labs(
        x="Mean ProteinGym correlation across genes",
        y=f"Mean UKBBGym correlation\nacross {scatter_df_small['n_genes'][0]} genes",
    )
    + facet_wrap('~readout_label', nrow=2, scales='free_x')
    + theme_minimal()
    + theme(
        figure_size=(12, 8),
        strip_text=element_text(size=14),
        axis_text=element_text(size=14),
        axis_title_y=element_text(size=14, lineheight=1.4),
        axis_title_x=element_text(size=14),
        # legend_position=(0.95, 0.05),
        legend_position="bottom",
        legend_box_spacing=0.01,
        legend_margin=0,
        legend_text=element_text(size=14),
        legend_title=element_text(size=14),
        legend_background=element_rect(fill="white", color="white", alpha=1),
        plot_background=element_rect(fill="white", color="white"),
        panel_grid_major=element_line(color="#cccccc", size=0.6),
        panel_grid_minor=element_line(color="#dddddd", size=0.3),
    )
)

plot.save(f"{FIG_DIR}/F4_proteingym_correlation_scatter.svg", dpi=200)

plot